# Imports

In [1]:
import numpy as np

# Functions

In [12]:
axial_power_factors = [0.38, 0.69, 0.93, 1.1, 1.21, 1.3, 1.47, 1.51, 1.49, 1.44, 1.36, 1.28, 1.16, 1.06, 1.01, 0.97, 0.94, 0.97, 0.96, 0.91, 0.77, 0.59, 0.38, 0.12]

def BWR_power_dist(
        BWR_thermal_power = 3926,
        N_Assemblies = 872,
        Length = 3.7,
        N_v_Slices = 24,
        axial_power_factors = axial_power_factors):

    power_per_assembly = BWR_thermal_power/N_Assemblies

    axial_power_factors = np.array(axial_power_factors)

    assembly_power_distribution = (axial_power_factors * power_per_assembly)/N_v_Slices

    return assembly_power_distribution

In [14]:
def BWR_channel_flow_area(
        channel_box_inside_width = 0.1335,
        D_fuel_rod = 0.01026,
        rod_pitch = 0.01295,
        N_fuel_rod = 92,
        N_water_rod = 2
        ):

    D_water_rod = rod_pitch + D_fuel_rod

    rod_area = (D_fuel_rod/2)**2*np.pi * N_fuel_rod
    water_rod_area = (D_water_rod/2)**2*np.pi * N_water_rod

    total_area = channel_box_inside_width*channel_box_inside_width

    channel_flow_area = total_area - rod_area - water_rod_area

    return channel_flow_area

# File generator function

In [21]:
def BWR_input_generator(filename = "BWR_INP_V_.txt",
        BWR_thermal_power = 3926e6,
        N_Assemblies = 872,
        Length = 3.7,
        N_v_Slices = 24,
        axial_power_factors = axial_power_factors,

        channel_box_inside_width = 0.1335,
        D_fuel_rod = 0.01026,
        rod_pitch = 0.01295,
        N_fuel_rod = 92,
        N_water_rod = 2,

        pellet_diameter = 0.00876,
        cladding_thickness = 0.0006,
        fuel_density = 10000,
        clad_density = 6500,
        fuel_conductivity = 3.3,
        fuel_shc = 300,
        clad_conductivity = 10,
        clad_shc = 500,
        fuel_clad_heat_trans_coeff = 9400,
        inlet_temperature_k = 552,
        coolant_mass_flux = 1193,
        system_pressure_MPa = 7.17):

    z_coordinates = [0.0771 + (0.2313-0.0771)*i for i in range(N_v_Slices)]

    ################################
    # Variable 1: Power Distribution
    ################################

    power_per_assembly = BWR_thermal_power/N_Assemblies

    axial_power_factors = np.array(axial_power_factors)

    assembly_power_distribution = (axial_power_factors * power_per_assembly)/N_v_Slices

    ###############################
    # Variable 2: Channel Flow Area
    ###############################

    D_water_rod = rod_pitch + D_fuel_rod

    rod_area = (D_fuel_rod/2)**2*np.pi * N_fuel_rod
    water_rod_area = (D_water_rod/2)**2*np.pi * N_water_rod

    total_area = channel_box_inside_width*channel_box_inside_width

    channel_flow_area = total_area - rod_area - water_rod_area

    ##############################
    # Variable 3: Wetted Perimeter
    ##############################

    wetted_fuel_rod_perimeter = np.pi * D_fuel_rod * N_fuel_rod

    wetted_water_rod_perimeter = np.pi * D_water_rod * N_water_rod

    channel_wetted_perimeter = wetted_fuel_rod_perimeter + wetted_water_rod_perimeter

    ##############################
    # Variable 4: Heated Perimeter
    ##############################

    channel_heated_perimeter = np.pi * D_fuel_rod * N_fuel_rod

    ###################################################
    # Variable 5-17: Fuel, Cladding and Core Properties
    ###################################################


    ##################
    # Format Variables
    ##################

    # Variable 1: Axial fission power distribution
    power_formatted = [
     f"{value:.4E}" for value in assembly_power_distribution
    ]

    # Variable 2: Channel flow area
    flow_area_formatted = f"{channel_flow_area:.3E}"

    # Variables 3 and 4: Perimeters
    wetted_perimeter_formatted = f"{channel_wetted_perimeter:.5f}"
    heated_perimeter_formatted = f"{channel_heated_perimeter:.5f}"

    # Number of fuel rods
    rods_formatted = f"{N_fuel_rod:02d}"

    # Variable 5: Pellet diameter
    pellet_formatted = f"{pellet_diameter:.5f}"

    # Variable 6: Cladding thickness
    clad_thickness_formatted = f"{cladding_thickness:.4f}"

    # Variable 7: Fuel density
    fuel_density_formatted = f"{fuel_density:05d}"

    # Variable 8: Clad density
    clad_density_formatted = f"{clad_density:04d}"

    # Variable 9: Rod diameter
    rod_diameter_formatted = f"{D_fuel_rod:.5f}"

    # Variables 10-14: Thermal properties
    fuel_conductivity_formatted = f"{fuel_conductivity:.1f}"
    fuel_shc_formatted = f"{fuel_shc:03d}"
    clad_conductivity_formatted = f"{clad_conductivity:02d}"
    clad_shc_formatted = f"{clad_shc:03d}"
    heat_trans_coeff_formatted = f"{fuel_clad_heat_trans_coeff:04d}"

    # Variables 15-17: Operating conditions
    inlet_temperature_formatted = f"{round(inlet_temperature_k):03d}"
    inlet_mass_flow_formatted = f"{round(coolant_mass_flux):04d}"
    system_pressure_formatted = f"{system_pressure_MPa:.2f}"


    # ##############################
    # Assemble COBRA Input File
    # ##############################

    lines = []

    # Initial empty line
    lines.append("")

    # Header
    lines.append("    0     2     2     2")

    lines.append(
        f"    2     1     0    {N_v_Slices:2d}"
        "     1     0     0    5      0     0     0     1"
    )

    # Axial geometry
    dz = round(Length / N_v_Slices, 4)

    lines.append(f" { -dz:.4f}")
    lines.append(f"   {N_v_Slices}")

    # Axial coordinates and linear fission power
    for z, power in zip(z_coordinates, power_formatted):
        lines.append(f" {z:.4f}")
        lines.append(power)


    # Channel geometry: Variables 2, 3, 4 and number of rods

    lines.append(
        f"     1  1.0   {flow_area_formatted}"
        f"    {wetted_perimeter_formatted}"
        f"     {heated_perimeter_formatted}"
        f"      {rods_formatted}        1"
    )


    # Fuel and cladding geometry: Variables 5-9

    lines.append(
        f"  {pellet_formatted}"
        f"       {clad_thickness_formatted}"
        f"      {fuel_density_formatted}"
        f"       {clad_density_formatted}"
        f"        {rod_diameter_formatted}"
    )


    # Thermal properties: Variables 10-14

    lines.append(
        f"    {fuel_conductivity_formatted}"
        f"       {fuel_shc_formatted}"
        f"          {clad_conductivity_formatted}"
        f"          {clad_shc_formatted}"
        f"         {heat_trans_coeff_formatted}"
    )


    # Remaining fixed BWR COBRA records

    lines.append(
        "     0     0     1     1     0     1     1     0     2     0"
    )

    lines.append("     0")
    lines.append("$            JSLIP")
    lines.append("     2     2     0")
    lines.append("     0     0     0     0     0     0")
    lines.append(" 10.   0.0   0.5  0.0")

    lines.append(
        "     0     0     0  0.0      2     0 1.30 1.30 0.0130"
    )


    # WERRX and damping parameters

    lines.append(
        "$                  WERRX DAMPP DAMPF                         EPSP ISCHEM"
    )

    lines.append(
        "     0     0    0.  0.     0.    0.  -.01     0.     0  0.     0.      2"
    )


    # Operating conditions: Variables 15-17

    lines.append(
        f"     1  {inlet_temperature_formatted}"
        f"  {inlet_mass_flow_formatted}"
        f"  {system_pressure_formatted}"
        "    -20     1   0.  0.0   0.0     0.   0.0  0.0"
    )


    # Final records

    lines.append("     0     0     0")
    lines.append("     0     0     0     0     0")
    lines.append("     0     5     3     0     0     0    -1")
    lines.append("$EOD")


    # ##############################
    # Write .txt File
    # ##############################

    with open(filename, "w") as file:

        file.write("\n".join(lines))
        file.write("\n")


    # ##############################
    # Return Results
    # ##############################

    print(f"Assembly power: {power_per_assembly:.4f} MW")
    print(f"Channel flow area: {channel_flow_area:.6f} m2")
    print(f"Wetted perimeter: {channel_wetted_perimeter:.5f} m")
    print(f"Heated perimeter: {channel_heated_perimeter:.5f} m")

    return {
        "filename": filename,
        "z_coordinates": z_coordinates,
        "power_values": assembly_power_distribution,
        "power_formatted": power_formatted,
        "power_per_assembly": power_per_assembly,
        "channel_flow_area": channel_flow_area,
        "channel_wetted_perimeter": channel_wetted_perimeter,
        "channel_heated_perimeter": channel_heated_perimeter,
        "inlet_temperature_k": inlet_temperature_k,
        "coolant_mass_flux": coolant_mass_flux,
        "system_pressure_MPa": system_pressure_MPa
    }

In [22]:
BWR_input_generator(filename="BWR_3_1.txt")

Assembly power: 4502293.5780 MW
Channel flow area: 0.009370 m2
Wetted perimeter: 3.11124 m
Heated perimeter: 2.96541 m


{'filename': 'BWR_3_1.txt',
 'z_coordinates': [0.0771,
  0.2313,
  0.3855,
  0.5397000000000001,
  0.6939,
  0.8481000000000001,
  1.0023,
  1.1565,
  1.3107,
  1.4648999999999999,
  1.6191,
  1.7733,
  1.9275,
  2.0817,
  2.2359000000000004,
  2.3901000000000003,
  2.5443000000000002,
  2.6985,
  2.8527,
  3.0069000000000004,
  3.1611000000000002,
  3.3153,
  3.4695000000000005,
  3.6237000000000004],
 'power_values': array([ 71286.31498471, 129440.94036697, 174463.87614679, 206355.12232416,
        226990.63455657, 243874.23547401, 275765.48165138, 283269.30428135,
        279517.39296636, 270137.6146789 , 255129.96941896, 240122.32415902,
        217610.85626911, 198851.29969419, 189471.52140673, 181967.69877676,
        176339.83180428, 181967.69877676, 180091.74311927, 170711.9648318 ,
        144448.58562691, 110681.38379205,  71286.31498471,  22511.46788991]),
 'power_formatted': ['7.1286E+04',
  '1.2944E+05',
  '1.7446E+05',
  '2.0636E+05',
  '2.2699E+05',
  '2.4387E+05',
  '2.